# Лабораторна робота №7 — N-грамне автозавершення

**Тема:** Система автозавершення тексту на основі N-грам.

Моделі 1–5-грам, оцінка наступного слова, ipywidgets-інтерфейс, перплексія на незалежному тесті.

**Особливість:** цей блокнот самодостатній: оригінальний корпус `shakespeare.txt` вбудовано, тож не потрібно нічого додатково завантажувати. Натисніть **Середовище виконання → Виконати все**.

In [ ]:
# 1. Установлення бібліотек для Colab / Jupyter
%pip -q install matplotlib ipywidgets

## 1. Дані корпусу та допоміжні файли
Відновлюємо оригінальний текст Шекспіра, статичну HTML-сторінку та автоматичні тести зі вбудованих стиснутих даних.

In [ ]:
import base64, zlib
from pathlib import Path
for rel_path, encoded in {
    'data/shakespeare.txt': (
        '...'
    ),
    'docs/index.html': (
        'eNqtW3tvG1d2/z+f4i6VLMmGw6coycOH63iTNIDjGkm3RSEI2SHnkpxoOMPODPWwIsCxHHsDGOtFYihot2nS/WPb/lVZfsmyJQH7'
        'CYZfwZ+k55x7Z3iHD0mLbQCL5H2ce+55/M5jJvVf/Orvr//DP9/6kPWCvt18p44fzDacbiM1XE/hADfM5juM1fs8MFi7Z3g+D2Ay'
        '6GgrqaYYdYw+b6Q2LL45cL0gxdquE3AHVm1aZtBrmHzDanONfuSY5ViBZdia3zZs3iiliHhgBTZv3vhsmX3Fwt+Hh6O74Wn4MjwI'
        'D8Pnozuj34bPw5PwZPSIwcTz8Hj09ejuaI/d1MKnozuw6g39O6oXBJ2YXcGYyf22Zw0Cy3UU3sI/hD+zt/f2l3WVzslon8HHafgM'
        'znkNP0pv73xfzbHwFY0ejvZh4Ul4Shzss9HXuCg8hAVnyCn8fS0Z3BfcHrDwRXgE94AvSBOpnNBuOHB0n5YBQZg9BqpAYLSHm1n4'
        '34IODOCRB3khKD/YFhdkTPdcN2A7cCPb9UCcPd7nOjMNb73GNK3V1ReK7VKxtIy/BobDbX2htFTi5VY8UIaRVnmpUsURbna5vlA2'
        'K+1qEX9bzrq+0Cl1ljoG/uwPA27qC62ldsUs4YDRboMc9YWVEm+3OzjSdW1YwVfapeWVGtslJv8GGGy5W5pv3bacrt5yPZN7GozA'
        'ArI5mAftuLattXjP2LBcT/f7cLFeRKHlmtuwqAN60zpG37K39U9AhV6O+dt+wPva0MppxmBgc00M5FKf867L2a8/SeV8w/E1n3sW'
        'MNg3vK7l6HC5ltFe73ru0DH1DcPLoLCyNSFHOQCXhxHbcrjW41a3F+ilfLUasZTf9IwB8EQWrfctJ1MqVYqDrRwDm25nSsXie0xj'
        'laXBVjYbn2sMAzcm4BgbsN+0/IFtbOsdm4M8DNvqOpoFN/B1FC33auzLoR9YnW1NWq3uDwzwpBYPNjl3aqxrDPRSeQCbB4ZpooDL'
        '8IsV43NanuGYkfQ2xU1WUL82D+AADenhNi1frFR5vyYWgrK4XlpBugk6sNqJDE4KSlhBNnExA9YEfCvQTN52PQP9TndchydFTAaV'
        'TZy4iCcKcWk27wR6uTjmwbjg5B73XFgipV0hObDyiiqcpSINVio4qBiBZ5iISF38BJoZbtvWwOfMCNgKqLL4Xm6hVK2YlSv4NQBR'
        'gCA8WMiqxffGDPBt3vLczUja4kqknIT0i9PSz5eWUfYkMyLfcb2+PhwMuNc2/Am5Tdy7V0oc2LaN/iBTqYIxLm1s5pbQBicNubg8'
        'S//VJeShb2wJoNavVIpjdeilJRJdwihsCA0TSomUOiazXK7GIpi2q3YPBD3lCWjWtAp/auhsOv6JbSNwB3p5cYqMNE+BMXoJGPZd'
        '2zKZYA3xDTiTCISqHvrSwCL7AE6ZMEKFXTIWcccFs8yL/Ep0rM/baNtjo6tWpfOxXnmGVsoVhIgKqIVVSkIvU1pYFEoQSIUCJ+nt'
        'skHeH7bYzmxZx0Ip1qRAIPp5VluVbNezTJAs/NUAYWAo4IAq9rDv+LrHB9wIMou5UsfLqqgiqQWWzXN5Chko30n0pIlYtJcTfMIK'
        'kD4QjjEMKYg/FXF7sSJvGy3i4AIYSW7aMOwhT2gDhZ90yuWV4gXKIPGuJAj3LECAWRqZcP/IRoWwL6kSCCrgQ5lirpQvV0ErLB7I'
        'L1fHWlpSpBjpJxajNB050auMDZUsqzJh6FcUWh2L26bCast22+vz3GKGIqRclzDUSAG23CBw+/ry+BTLGQyDnM9t8KPpsxQzWyh2'
        'Ssvl8qwQLUAGI+4M+1tYLFZLSytTtldSnR5voni9bjkQSaygxtxhgLgpQ5fCst5x20NfMi5+sB15xEykpgSoZ5juJkkeIhAc6XVb'
        'RqZUvpIrV5Zy5eJyLl8qZ2uq/KdsZbapgNMy+CctoqqCdjWRDDhuAHn4Jf0nIlEh3B8T8YfdLvcR96ZxewKukR8yKrDdKPosE38i'
        '1YmTmsDwgjjdG4KZONExyOzQ84HbgWuJlCjS2xVUm4jkSe0WJ6L7QrlYMSrtWdYzbTGVpaWlamfSFGaxpvfcDe4l8XChfKW6VF2M'
        'WTovX4lu6PcN256X26jJUOTO0flkeNqG5Vstm+9E1lqeQF9MymOz2rBun+9oHfpvSqTkHaqrRdYhsqwEDJlGYGiB0UJIV7aMJWIb'
        'kFvp0ZfabmJLYOYSP3s7sZ8WI4VTnkQmpKNganFRQQAzO/xMnqLbhh9okDfYkwcqMzvKSR6a7ySZ3kX4j+wmALEYh2hn2G+h/dA0'
        '7LcM+IRBsLi2DvSHtuHhbx8P7cFuGynszDKTyfwyAhGqD6V0MH7NicxqwABtLqK2L77WLhuC2cq9Ml2ncAfZJtuRBkJZIwy1XZOf'
        'n7//bZ9DEp4ZJ44rmJtl2U6Uy+ycm7uUKXfZjaLszlyo3KVCIeYbxYKJ7q6oXHZUh1uJlBfH8t3daT6ri8jnjoTs8w4+/yISxncp'
        'uxhbfRV5UzKZnTFHmB1JvlWjjdwb49YuCrdekF2DekF0c+pYU1M7wbQ2GCSnvt9IIWSnRGehjjTlMHxNMbROjbKvRir8Edsyo/3w'
        'adS/SDVVOlQqppo365iONz/+7NqncD5+ZQV249oHrLhcL8By2tOsG6zn8U4jtWDyvptqhv8Zno3uhKfhk9Ee9kxGd7GrYyjrPO4P'
        '7cCHpY+xC0Ptl2fhi6hBlFjcC4KBrxcKXSvoDVtgG/1Cy+2ahgMBad0FpK5qXatTuPHZsnYTir++dg1qdFg1gFSQp5ofW8HfDVtE'
        'UXBcAGFICaEgwXvlndGkklKQxSCw+R9wF7pReIxdHrzeEQ08oz7SD6LddBgesT+/ZOEfQLhPqBGEdz+lZtMBkwROqZ+EPSvBELGC'
        'zJSaqBXRFQvPYMtz6kwR3VP29s5jRmN4CkwfwNHHspmFWx7mwTBKMbVBdAes7+bJGekf4W5qaR0xsolLNe1EswtJnggeGPB8gvYk'
        'WnA5BgJTflGn7Gj8e1ETbbVq3LTLs/C/xO1JQkdzu3XXTFNbP69VB+eeILNE9xhpwV1O/urWXb0wiKWr2AhVrWA25B23toOe60hX'
        'EUMXtiUTq+lyTPTOADkSc9ZgG8CqywM/MXyLe2DqW1awHQ2PzUpgBfea74ifUbFrmY1U7IP1XrkZ/gwsvQQdvh49RCGi1FgkABLl'
        'EXD+HAU72gOq5ebYwqCmBQP7jkzrGNRxQPc7gR2vyGhQ2KxcfA9dw+fcZItl8pI3pKh96vvCscJHHig2cMBo4XqjmKvOkb5E41Q0'
        'l5xFxE16NMFfqinh70F0MtrvndEjusDRGNmiXYTZKZJa4BmWk2qCN04vw1oy1cRmU8KW9iLgUbz9snz+UYgv8rFL8giJ5bkslv8/'
        'WfyZ8OINeskYr1D7gJTnsrnhto3WuXyGL9kv7aA2dNZ/2YUEnkwCBwo+/v4reEb1v5IG/VsC9GkwOZf31gUijhcF2wOuLkxAvuqo'
        '0jVneaqIquSm+xc8hZjpmj8KfwblHOO9JdyJEH2GNEhlB8INIg0eAGh/iwELz8J1GCjicK5EAwDuP2FYekJc7IEtPJcBL2ZShBIB'
        '0aBTnATxw9EgfAyjqIZ9/C6i0DHdKz5APBaqMeL1MAKW2SCfnBrdAxqvRl+zEcTOQ3bdBf3PxXGRwc1BEuq+KHMYryvNSz/kArVU'
        'EptFE0wSp6QzBdWqh2w4WK8goCJFECGF5RNI1DB4w2UAr44TqQFB9ClM7wGgxRkUndCsU6+DrCiizMiMwTt6nK1DjIHEUEmYGim3'
        '00leVBGDSI9TMu2bewkqVuAKPyHEEGSdhsfKA7qYO9k2QvbknrpLj/giJkupZolSn2R6gQAoFk5uKAO6iVzpyWWWV1JM8MDNZkUc'
        'lMhU5u9cTDUXacOidvHiaqpZpcXVmYsLgodJLBOyP0fOA0iQrS0p6Aci2QRBo38paeQp2M6p4tgzbEMSYlBytHnPtUEVskYAlMjn'
        'mO3OtJKZCDzDYiLEpWAfZ3To8pgRJbhNQNC5lJUeloDbxIAodqwNYHUAxTJWAuEP5KUHhA4HShas5mRv7/xp+tQYT0XzLRWjsmY5'
        'HYTmMaQkUP2SKPK/oLPRfcTQM4nkvxs9noKMAdZV42yd6omD0Tfg9Y9zKLNjkO5rAocjmoJ0bh9ATykhMMET4QJEj+byGOsGGHnE'
        'KMvF+gABmjK4g6iwOVMqB4G/MhPPySfjpxh6QJC/Q5R9FZnbnkT+E6wWXk3oPSpYBhM3rPuB5zpdVNVTmUm+EEFHqoryYyyFaZmI'
        'JM/wJlQb4WEkAzxr3rGKfVFl8gglJ0orEUkxDYDR1yTsRxg4RESCdRCq4M7oUm+Q/kSgGu2fc6UPjPY6OI3Cu8g5no++Ies7It5Q'
        'xlFqBikwhBMhzqmj9iTn3+LkGa0FrUJ5KbSmVB1HQjRvSEpwdzo0fDaDe7xpFLIVl5i+1IQ7NMP7s4IxShXTBeRLOB2wghUw5l0v'
        'RNBHa3siWXohq4Y3Ir2AnEMrFxPaQgLPyEjjOuFo9M3Mm9SkR1HFnEgcEikqExWbNv++f2nG1pyZVMxAAXL9n2YUsVQVKrES1tWt'
        'fjfOQq3bELS8NhCKK7/8gOK4jW/EfE+4cA+EcDwrr0WPTo8eCW3Et8Y8KQUwb2C/CvPk29up5gxF/whCe4Fphoixx3TWt+FzSAIf'
        'C/RAWzskkkekl6djg5a1BltMXBD/KXWfTCSB9B56NalONFGEgU45AqkrqaU5wv4jRUlR7E2Xu0LQoicsN4+7xEAhoLZbPfDwazP8'
        '99hgHtYLMECDs9T59v73YkEBtxYiMvRGDMXeQXxEgUbxEwdmyR/7UA8olX9N3nKKkhPu85JRgk7A9JoEf0BdqntUaB/ObKeg2kU5'
        'EBXqoosFTiVyCToEokwOXRmKewD5B3QspvFv6IRjOh8Qg1EIOoFo8IgEPM5GhYpm2rCDLbsv2u7QCXzViv8timZj2BYZ4HHcLUAu'
        'xzg3bb1KcnIpZ1VN5X/wim/E2RKa1fadsJUh5FC21Uz0rRIJjKiCjyg8oyfeJySU3IvuiJQxq2NzvalUu/UCjcgYK1wWA0TCZZX7'
        'g4yBF+LnJ9l6FfH6QInFAi7nGQKyfk/IPdkeO5O9HcgpRIL5UtTMUdJ5LKxGADQgaBwLYEDh6/5UXSZurTa25KX32Meu2wVPpJqN'
        'rO2Y4u4ZsaFEAsGcrBNFJUsZkHynLyGWaecUdSpiFmY9rykKoZW9SjiEGIhDjghcSUybbBcydKynYyhOCCLq6oBlRPEHRfgsx266'
        'AW+57nqOXf/8H6XuX5Kq7qJQyNHPqOA8ikBx9IOkXEB7nDR3snZ8hjTuc+MvgJJLNqmxA6f2MDGentcmhuXlYnmJOny/F3WrTCei'
        'CKxaCAI4nUwTikghoMxsSY7VdCD8AZXzTLCNqQT66t3Ro9ykfug+6xqsmZlYggiFkOiBShRM6uKVUyHF9NDnzMd2Y5Cu0QhU1H7A'
        '3mUNwHHWaDLTbQ/73AnyYMof2hy/frD9iZmxzKzYAKUT2+StT8HGbdjlDG1bpdTpBzC6haQcvsk+cQI7f5MeMH7ken0jyKSH69qv'
        'r6Vz+GRuy+oP+x95Bun5V1bXCny9vJvNd8TSLXlkZ+iIRpIDtf/n4wopk92R0dLqZH4RMZX1eDD0nJqcEmx5/F+GsIvDHZlgJ/Nu'
        'Jk31ejornmdlc0yUkLAE5sT3aDIfuDfcTe5dN3yeyeZBgv1MNnlE4K5zx4fNSFk2KuZsh8u1e5nCqqHdXns/c1VfTb+9869r4mf2'
        'aqGbZV99xVbXcOEgs9lobsZHofRty0cZr67lGKgTbxTfLloGAmQZXOs0xnPMaTZK8FfTsmwnzg0F8z2g6XrbqNBGo1G6mk7r1zzP'
        '2M58agS9vHjfx9FKmrhk3uZON+hlQVOWbWfSdb8JNwVKbdCaXOLbEO0zWgZ2ZWHhl67lZNIsHd8kOhkMzLM4yi1SYL6Pf/3Vz2HC'
        '6Wac7NrV/KpkcE1IZkzE6rCMJBFxxXYiEcmJWiQnp8ZaHjfWo0e9LP6UG/AD7wRORFS3G036WC2u5enlEP+frKCXEbYBtxKXLOaq'
        '8bVA90oND1LxODUkruMTUY87Y6shHQkZrG6CJaL5ua015naIjeykilqBAxzG/tmGiwRcumgmLV4JSeOrJIGTx45tIxoTQwScN/E1'
        '9IjB9KQqNhvzqOMzGaC9mUejvi5fXEemJ0kM5pPAt0uAxiBBQ/oi3hy8yv3I2uJmZlExEmTdGAy4Y2Y2c4OJCTA4UMB6I5NtNHfY'
        'tNs1Znkiue77YIrv4wXwS23K3RtpkNoU2tSSR9BrLzi6O+ZqSv2Sd+A25n1XAS0yuMhupzarkkpD3RJVwErfG4PofYrtUS5xKJOc'
        'g7htpmY5GEbiajSfVsw27gUlDwabQ+8BVBhj6FX2m/A7URtReh4nLuzdHVy8q1Tu+d8wHZaLygpYfTi3GyCeCk/TbImugw7EYxbU'
        'E6A6+W7mweJyQtzgbdLZIM6Bi61K8M/FCs1FNrCGmoB4lzdM88MNEMENUBJ3wEwtFIPceDXd7hlOl6f1NPUg07kJe5H6vuW5fcvn'
        'ebD+zGqHI/Cn5RO//Jc+umweyikn44EJo0V4eXc9G/Tw/W0MoB96HnAe7UD/FuGNebQ7k93N5iJAkcQJQ/8C0rgeCE/RXZPbM6vy'
        '9ByjtWvobZHFR6DdoKnYEcCe6CnjhBFDdpCJLk/zX/j4/xc4be4L78LHfuftgWl1i3IcPYk7ZyvN46tM21/gOyvjvRJdgXJjNZ+P'
        '3o3Jx40JCCAA/q4XZDJGrgV3N8Zz21pL+ZGFMKGy1Jp9GRxWd9WitYjaE+tpbVK0StxATYItz2I5Sy85UV7izcXkwFPjsVxvzF9u'
        'prO51rnTNSPBPjAouW9NiQHnVCmM+cCYjrOYipAAnGxLjV/xO2jpWuBFCIu6ISCXrYgx9gZedhzwZ6B6BMuQwVBeRtHkYixGuHpG'
        'mfyp6LG+nGjNH030p0YPoSD4SXY6RDf2gB6GPRKPKZl4zYfdMuBY7DJC3g/VwGsqveTTTdm1OKMOL9aIBdCGn0/v0jWgZpLJfr0g'
        'uzDi/6b7Py2N4N8='
    ),
    'tests/test_lab7.py': (
        'eNqdVs9uIzUcvucpTC7xSO7QtCqLIuVQ2goQ3S6CLpcQWc7Ek1j12LO2J2m62sMCd25w4IJ4Ag4gITjwCtNX4En42TPTTSZJi4ha'
        'zYzn9/fz9/vG3W63/KX8/f7t/TflH6j8rfzr/ntU/lT+jP757odnMSp/LH8t/77/9v5t+ecA5Ss31wodZKhQwjluHZoKm+gFN+jA'
        'Ir9g0cGi2+12RJZr41DG3Ly5b3w6qdEZSrSUPHFCK4tqgzNdKMdN9T4HTykmzbvP1wLZle3Af+xNYqEsNw4fEmSdwd4MU5oKySmN'
        'YsOtlguOI7A1XDk76o+jqEog2eRZEx1/eX36xTVBF1fnBL28+oygGXfU6RuuxB2f0ilzjEDJUB9dajO1pIO2ft4lvKRL4eZU5bKw'
        'NDX8VcFVsiLI8FyyhFOtF7XdZEULdbMrVpVK0ZlhmSXoyl8vmZoVbMaf6ymXBAGSAuDlNDd6wiZCCreKOp3z0+tTNESjUU/0COpJ'
        '2Bx/TZizvTFBreWpnq0vW94yXumiR2rrennc+erF2elHkMMCdHyKLXd4iVJtkAWIkVAo1OAXlv7JrwLonUQya9G1JwluuBD7xzNm'
        'eTQIKEx5CvbuZQ5RZVov+p9/jDNIug1FnAqHfUqCQmUE3QwP4xPI2ET0mZrdZJ5yO6NDdcCki1cFk3h7+3H3k4vLyxcEuTk3/L2v'
        '1blWPYdmOu5GxMM951JqQCm8D+h1p96kC4DONODWLshT48lC1jgXeoxGD7tzHO1yu9LuU4WrjSVPcLJG7ShqlwasfLKyfXTG77BA'
        'Pc6Mm0PvnkvV4ngDLT9s28gE1rcLCItAgI3RwBXPK4aO94JSb6r3GDWzXl/APQK//tN+G5MDSlG7tVhmmLLCc4wqbTIm9zDOT4fy'
        'kgKKdxuGZoT7ZATo4CPie/J3x2S9O1g4gYVUexDHgw3VcNox6UeyyHA1KvGaLOAlqRMRFa0PZjBc6IRNCsnMKtqIuYbDqcy0rdEI'
        'qUifhN23w/5hG4KJ8ICBLPGUG0v1BCIsgk7sI9THhoOQmV2V13gHRMgRcGeXTUX2xmZ7SzYL8vPzP4sJo7exKeR4T01BSluW22MG'
        't0KBwvgBohnLLajOf5q+XSk9Mx5Hyn/a9sFki9nMf1PggxygErftIpzOCWKJKyqqVeFrN5gOVG2VGh4TsKQ3cLUO5j9Iz7CX9B4Z'
        'MXAYHY7hr/n67LS9NgXHTEq8jENk6yNjH7kiNaGe1hAreiRV1UBQihZvWXKj07TdNX2k50LZnCciFXyK7u7ufPMnT6feEo2cm1zy'
        'W9ghmmuvHnBk2aUXvjvQlxkHqfggGuyb1prBm0R5SFGJvopIPz7cO/AB6SycrywwVDi+P9AWk9YYR8PMwbdmqyHV6Hl96sOv1wU2'
        'GhyR8BzmKBr037wr1YcHv13nn0YuUPCFGXj9htSZCDrxB4PdOr+ubz4csOP9D6EtkSJKFcvgMImGQ9SjMKNCUdqr+ng4xfhVHHX+'
        'BZZxnXo='
    ),
}.items():
    path = Path(rel_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(zlib.decompress(base64.b64decode(encoded)))
    print(f"Відновлено: {rel_path} ({path.stat().st_size} байт)")

## 2. Реалізація моделі N-грам
Код далі є **тим самим**, що у `lab7.py` репозиторію. Для 1–5-грам використовуються окремі частоти переходів, Add-k згладжування та обчислення перплексії в логарифмах.

In [ ]:
%%writefile lab7.py
"""ЛР №7 — N-грамна система автозавершення (1–5-грам).

Запуск: python lab7.py --corpus data/shakespeare.txt --output results
Навчальна модель: add-k згладжування, однаковий train/test split, log-perplexity.
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable

START, END, UNK = '<s>', '</s>', '<unk>'
TOKEN_PATTERN = re.compile(r"[a-z]+(?:['’][a-z]+)?", flags=re.IGNORECASE)


def load_data(file_path: str | Path) -> str:
    """Читає вихідний UTF-8 корпус."""
    return Path(file_path).read_text(encoding='utf-8')


def split_to_sentences(data: str) -> list[str]:
    """За методичкою кожен непорожній рядок корпусу = окрема послідовність."""
    return [s.strip() for s in data.splitlines() if s.strip()]


def tokenize_sentences(sentences: Iterable[str]) -> list[list[str]]:
    """Англійська токенізація, збереження внутрішнього апострофа, lowercase."""
    return [tokens for sent in sentences if (tokens := TOKEN_PATTERN.findall(sent.lower()))]


def get_tokenized_data(data: str) -> list[list[str]]:
    return tokenize_sentences(split_to_sentences(data))


def count_words(tokenized_sentences: Iterable[list[str]]) -> Counter:
    return Counter(word for sentence in tokenized_sentences for word in sentence)


def get_words_with_nplus_frequency(tokenized_sentences: list[list[str]], count_threshold: int = 2) -> list[str]:
    if count_threshold < 1:
        raise ValueError('count_threshold має бути >= 1')
    return sorted(w for w, c in count_words(tokenized_sentences).items() if c >= count_threshold)


def replace_oov_words_by_unk(tokenized_sentences: list[list[str]], vocabulary: Iterable[str], unknown_token: str = UNK) -> list[list[str]]:
    allowed = set(vocabulary)
    return [[w if w in allowed else unknown_token for w in sentence] for sentence in tokenized_sentences]


def preprocess_data(train_data: list[list[str]], test_data: list[list[str]], count_threshold: int = 2):
    """Словник ТІЛЬКИ з train; ніякого витоку тестових даних."""
    vocabulary = get_words_with_nplus_frequency(train_data, count_threshold)
    return (replace_oov_words_by_unk(train_data, vocabulary),
            replace_oov_words_by_unk(test_data, vocabulary), vocabulary)


def count_n_grams(data: list[list[str]], n: int, start_token: str = START, end_token: str = END) -> Counter:
    if not 1 <= n <= 5:
        raise ValueError('n має належати 1..5')
    counts = Counter()
    for sentence in data:
        seq = [start_token] * (n - 1) + list(sentence) + [end_token]
        counts.update(tuple(seq[i:i+n]) for i in range(len(seq)-n+1))
    return counts


@dataclass
class NgramLanguageModel:
    """Власна N-грамна модель із згладжуванням Лапласа/Add-k."""
    vocabulary: list[str]
    counts: dict[int, Counter] = field(default_factory=dict)
    history_counts: dict[int, Counter] = field(default_factory=dict)
    k: float = 0.5
    _vocab_set: set[str] = field(default_factory=set, repr=False)

    @classmethod
    def fit(cls, train_sentences: list[list[str]], vocabulary: list[str], k: float = 0.5):
        if k <= 0:
            raise ValueError('k має бути > 0 для оцінювання перплексії')
        allowed = set(vocabulary) | {UNK, END}
        if any(START in sentence or END in sentence for sentence in train_sentences):
            raise ValueError('Спеціальні маркери в train data використовувати не можна')
        words = sorted(allowed)
        model = cls(vocabulary=words, k=k, _vocab_set=set(words))
        for n in range(1, 6):
            grams = count_n_grams(train_sentences, n)
            model.counts[n] = grams
            # Знаменник із тих самих n-грам, що й чисельник:
            # точна сума виходів з історії, без підрахунку непарних EOS.
            model.history_counts[n] = Counter({(): sum(grams.values())}) if n == 1 else Counter()
            if n > 1:
                for gram, c in grams.items():
                    model.history_counts[n][gram[:-1]] += c
        return model

    def normalize_history(self, context: Iterable[str], n: int) -> tuple[str, ...]:
        if not 1 <= n <= 5:
            raise ValueError('n має належати 1..5')
        if n == 1:
            return ()
        seq = [w if w in self._vocab_set or w == START else UNK for w in context]
        return tuple(([START] * (n-1) + seq)[-(n-1):])

    def probability(self, word: str, context: Iterable[str], n: int, k: float | None = None) -> float:
        smooth = self.k if k is None else k
        if smooth <= 0:
            raise ValueError('k має бути додатним')
        if word not in self._vocab_set:
            word = UNK
        history = self.normalize_history(context, n)
        observed = self.counts[n].get(history + (word,), 0)
        total = self.history_counts[n].get(history, 0)
        return (observed + smooth) / (total + smooth * len(self.vocabulary))

    def suggest(self, context: str | list[str], n: int = 3, top_k: int = 5,
                start_with: str = '', k: float | None = None, backoff: bool = True) -> tuple[list[tuple[str, float]], int]:
        """Пропозиції з backoff при небаченій історії; повертає використаний порядок."""
        if isinstance(context, str):
            tokens = TOKEN_PATTERN.findall(context.lower())
        else:
            tokens = list(context)
        order = n
        if backoff:
            while order > 1 and self.history_counts[order].get(self.normalize_history(tokens, order), 0) == 0:
                order -= 1
        prefix = start_with.lower()
        candidates = (word for word in self.vocabulary if word not in {END, UNK, START} and word.startswith(prefix))
        scores = [(w, self.probability(w, tokens, order, k)) for w in candidates]
        scores.sort(key=lambda p: (-p[1], p[0]))
        return scores[:max(0, top_k)], order

    def perplexity(self, test_sentences: list[list[str]], n: int, k: float | None = None) -> float:
        """PP = exp(-Σ log P(w|history) / M), M — кількість токенів разом з EOS."""
        neg_log_likelihood = 0.0
        num_predictions = 0
        for sentence in test_sentences:
            history = [START] * (n-1)
            for word in list(sentence) + [END]:
                p = self.probability(word, history, n, k)
                neg_log_likelihood -= math.log(p)
                num_predictions += 1
                history.append(word)
        if not num_predictions:
            raise ValueError('Тестова вибірка порожня')
        return math.exp(neg_log_likelihood / num_predictions)


def estimate_probability(word, previous_n_gram, n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0):
    """Функція у формі, близькій до методички: попередні n слів і (n+1)-грам."""
    if k <= 0 or vocabulary_size <= 0:
        raise ValueError('k і vocabulary_size мають бути додатними')
    history = tuple(previous_n_gram)
    numerator = n_plus1_gram_counts.get(history+(word,), 0) + k
    # Знаменник рахуємо лише з дійсних виходів, як у NgramLanguageModel:
    ngram_history_frequency = sum(c for gram, c in n_plus1_gram_counts.items() if gram[:-1] == history)
    return numerator / (ngram_history_frequency + k * vocabulary_size)


def _write_csv(path: Path, rows: list[dict], columns: list[str]):
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        wr = csv.DictWriter(f, fieldnames=columns)
        wr.writeheader()
        wr.writerows(rows)


def make_website_model(model: NgramLanguageModel, path: str | Path, limit_per_n: int = 1000):
    """Компактна реальна модель для статичного GitHub Pages: топ-контексти і слова.

    JS-додаток використовує лише підмножину контекстів; Python/Colab — повну модель.
    """
    data = {'source': 'shakespeare.txt', 'k': model.k, 'vocabulary_size': len(model.vocabulary), 'models': {}}
    for n in range(1, 6):
        histories = [()] if n == 1 else [ctx for ctx, _ in model.history_counts[n].most_common(limit_per_n)]
        rows = {}
        by_history = defaultdict(list)
        chosen_histories = set(histories)
        for gram, c in model.counts[n].items():
            if gram[:-1] in chosen_histories and gram[-1] not in {END, UNK}:
                by_history[gram[:-1]].append((gram[-1], c))
        for history in histories:
            # Використовуємо лише слова, які реально траплялися в контексті,
            # а для пустих історій JS знизить порядок до 1-грам.
            observed = by_history.get(history, [])
            observed.sort(key=lambda x: (-x[1], x[0]))
            total = model.history_counts[n].get(history, 0)
            scores = [[w, round((c+model.k)/(total+model.k*len(model.vocabulary)), 7)] for w, c in observed[:20]]
            rows[' '.join(history)] = scores
        data['models'][str(n)] = rows
    Path(path).write_text(json.dumps(data, ensure_ascii=False, separators=(',', ':')), encoding='utf-8')
    return data


def evaluate(corpus_path: str | Path, output: str | Path = 'results', count_threshold: int = 2,
             k: float = 0.5, seed: int = 42, train_ratio: float = 0.8,
             web_model_path: str | Path | None = None):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    sentences = get_tokenized_data(load_data(corpus_path))
    if len(sentences) < 10:
        raise ValueError('Потрібно не менше 10 непорожніх рядків з англійськими словами')
    rng = random.Random(seed)
    rng.shuffle(sentences)
    pivot = int(len(sentences) * train_ratio)
    if not 1 <= pivot < len(sentences):
        raise ValueError('train_ratio має створювати непорожні train/test')
    train_raw, test_raw = sentences[:pivot], sentences[pivot:]
    train, test, vocab = preprocess_data(train_raw, test_raw, count_threshold)
    model = NgramLanguageModel.fit(train, vocab, k=k)
    load_time = time.perf_counter()-t0
    rows = []
    for n in range(1, 6):
        start = time.perf_counter()
        pp = model.perplexity(test, n)
        duration = (time.perf_counter()-start)*1000
        rows.append({'n': n, 'model': f'{n}-грами', 'perplexity': round(pp, 5),
                     'ngrams_unique': len(model.counts[n]), 'evaluation_ms': round(duration, 3)})
    examples = ['the king', 'my love', 'i shall', 'to the', 'and the', 'you are']
    samples = []
    for phrase in examples:
        for n in [2, 3, 5]:
            top, used_n = model.suggest(phrase, n=n, top_k=5)
            samples.append({'context': phrase, 'requested_n': n, 'used_n': used_n,
                            'suggestions': [{'word': w, 'probability': round(p, 6)} for w, p in top]})
    # Для перевірки нормування умовних імовірностей.
    norm_sum = sum(model.probability(w, ['the'], 2) for w in model.vocabulary)
    if not math.isclose(norm_sum, 1.0, abs_tol=1e-9):
        raise AssertionError(f'Ймовірності не нормовані: {norm_sum}')
    oov_test = sum(w == UNK for sent in test for w in sent)
    total_test_tokens = sum(len(sent) for sent in test)
    metrics = {'corpus_name': Path(corpus_path).name, 'seed': seed,
               'train_ratio': train_ratio, 'min_word_frequency': count_threshold,
               'smoothing_k': k, 'train_sentences': len(train), 'test_sentences': len(test),
               'train_tokens': sum(len(sent) for sent in train_raw),
               'test_tokens': total_test_tokens,
               'vocabulary_size': len(model.vocabulary), 'rare_unk_count_test': oov_test,
               'test_oov_rate': round(oov_test / total_test_tokens, 6),
               'training_seconds': round(load_time, 3), 'perplexities': rows,
               'best_model': min(rows, key=lambda r:r['perplexity'])['model']}
    (output/'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
    (output/'suggestions.json').write_text(json.dumps(samples, ensure_ascii=False, indent=2), encoding='utf-8')
    _write_csv(output/'perplexity.csv', rows, ['n', 'model', 'perplexity', 'ngrams_unique', 'evaluation_ms'])
    report = f'''# Звіт: Лабораторна робота №7

**Тема:** Система автозавершення тексту на основі N-грам.

**Корпус:** `{Path(corpus_path).name}` (англійський текст Шекспіра, наданий у завданні).

## Методика
- Текст поділено за непорожніми рядками (кожен рядок — послідовність).
- Токенізація англійських слів (нижній регістр, без пунктуації, апостроф у слові зберігається).
- Розбиття на навчальну ({train_ratio:.0%}) та тестову ({1-train_ratio:.0%}) вибірки із seed={seed}.
- Словник формується **тільки за навчальною вибіркою**; слова рідше {count_threshold} разів замінюються на `<unk>` в обох вибірках.
- Моделі: уніграми, біграми, триграми, 4-грами, 5-грами. Маркери початку `<s>` і кінця `</s>`.
- Обчислення умовних ймовірностей: **Add-k**, k={k}. Перплексія рахується за log-імовірностями лише на відкладеному тесті, включно з кінцем речення, без `<s>`.
- Для автозавершення підтримуються top-5 пропозицій, фільтрація за першими літерами, backoff до коротшого контексту.

## Показники даних
- Навчальних рядків: **{len(train):,}**; тестових рядків: **{len(test):,}**.
- Навчальних токенів: **{sum(map(len,train_raw)):,}**; тестових токенів: **{total_test_tokens:,}**.
- Словник включно зі службовими `<unk>` та `</s>`: **{len(model.vocabulary):,}**.
- Частка невідомих слів у тесті: **{oov_test / total_test_tokens * 100:.2f}%**.

## Перплексія на тестовому наборі
| Модель | Перплексія ↓ | Унікальних N-грам | Час оцінки, мс |
|---|---:|---:|---:|
'''
    for row in rows:
        report += f"| {row['model']} | {row['perplexity']:.3f} | {row['ngrams_unique']:,} | {row['evaluation_ms']:.1f} |\n"
    report += f'''\n**Найменша перплексія:** {metrics['best_model']}. Вона свідчить про найкраще середнє передбачення відкладених токенів у цьому експерименті, але не гарантує найкращу якість будь-якого завершення тексту.

## Приклади передбачень
'''
    for sample in samples:
        report += f"- `{sample['context']}` ({sample['requested_n']}-грами; використано {sample['used_n']}): "
        report += ', '.join(f"{a['word']} ({a['probability']:.4f})" for a in sample['suggestions']) + '\n'
    report += '''\n## Висновок
Реалізовано всі п’ять порядків N-грам, згладжування Add-k, пошук наступних слів, інтерфейс на ipywidgets та оцінювання перплексії. Показано залежність якості від довжини контексту та розрідженості 4–5-грам. Результати залежать від корпусу, порога частот і згладжування. Статична вебдемонстрація використовує лише найбільш поширені контексти; повна модель доступна у Python / Colab.\n'''
    (output/'REPORT.md').write_text(report, encoding='utf-8')
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot([r['n'] for r in rows], [r['perplexity'] for r in rows], marker='o')
        ax.set(xlabel='Порядок N-грамної моделі', ylabel='Перплексія (нижче — краще)', title='Порівняння N-грамних моделей на тестовому наборі')
        ax.set_xticks([1,2,3,4,5]); ax.grid(alpha=.25)
        fig.tight_layout(); fig.savefig(output/'perplexity.png', dpi=150); plt.close(fig)
        fig, ax = plt.subplots(figsize=(9,5))
        ax.bar([str(r['n']) for r in rows], [r['ngrams_unique'] for r in rows])
        ax.set(xlabel='Порядок N-грам', ylabel='Кількість унікальних N-грам', title='Розмір моделей')
        fig.tight_layout(); fig.savefig(output/'ngram_counts.png', dpi=150); plt.close(fig)
    except ImportError:
        print('matplotlib не встановлено: PNG-графіки пропущені')
    if web_model_path is not None:
        make_website_model(model, web_model_path)
    return metrics, model


def create_ui(model: NgramLanguageModel):
    """Інтерактивна форма для Jupyter / Google Colab (вимагає ipywidgets)."""
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    input_text = widgets.Text(value='the king', description='Текст:', placeholder='Введіть англійський текст...',
                              layout=widgets.Layout(width='620px'))
    prefix_input = widgets.Text(value='', description='Початок:', placeholder='Перші літери (необовʼязково)',
                                layout=widgets.Layout(width='620px'))
    order = widgets.Dropdown(options=[(f'{i}-грами', i) for i in range(1, 6)], value=3, description='Модель:')
    smooth = widgets.FloatSlider(value=model.k, min=.05, max=2.0, step=.05, description='Add-k:', continuous_update=False)
    button = widgets.Button(description='Запропонувати', button_style='info', icon='search')
    output = widgets.Output()

    def append_word(w):
        input_text.value = input_text.value.rstrip() + (' ' if input_text.value.strip() else '') + w + ' '

    def update(_):
        with output:
            clear_output()
            suggestions, used_order = model.suggest(input_text.value, n=order.value, top_k=5,
                                                     start_with=prefix_input.value.strip(), k=smooth.value)
            if used_order < order.value:
                print(f'Невідомий контекст: використано backoff до {used_order}-грам.')
            if not suggestions:
                print('Немає слів із таким початком.')
                return
            buttons = []
            for word, prob in suggestions:
                btn = widgets.Button(description=f'{word} ({prob:.4f})', layout=widgets.Layout(width='auto'))
                btn.on_click(lambda _, w=word: append_word(w))
                buttons.append(btn)
            display(widgets.HBox(buttons, layout=widgets.Layout(flex_flow='row wrap')))

    button.on_click(update)
    display(widgets.VBox([input_text, prefix_input, widgets.HBox([order, smooth]), button, output]))
    update(None)
    return {'text': input_text, 'prefix': prefix_input, 'order': order, 'k': smooth}


def main():
    parser = argparse.ArgumentParser(description='Лабораторна №7 — автозавершення тексту N-грамами')
    parser.add_argument('--corpus', default='data/shakespeare.txt')
    parser.add_argument('--output', default='results')
    parser.add_argument('--website-model', default='docs/model.json')
    parser.add_argument('--k', type=float, default=0.5)
    parser.add_argument('--threshold', type=int, default=2)
    args = parser.parse_args()
    path = Path(args.website_model)
    path.parent.mkdir(parents=True, exist_ok=True)
    metrics, _ = evaluate(args.corpus, args.output, k=args.k, count_threshold=args.threshold, web_model_path=path)
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
    print('Результати збережено у', args.output)


if __name__ == '__main__':
    main()


## 3. Автоматична перевірка основних функцій
Перевірка токенізації, `<unk>`, згладжування, нормування умовних імовірностей, автозавершення, backoff та перплексії.

In [ ]:
import unittest
suite = unittest.defaultTestLoader.discover('tests')
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), 'Автоматичні тести не пройдено' 

## 4. Навчання і оцінка на корпусі
- 80% рядків на навчання, 20% — на тест (seed=42);
- словник за train, поріг частоти 2;
- маркери початку/кінця речення;
- перплексія для кожної моделі 1–5, **без використання тесту для навчання**.

In [ ]:
from pathlib import Path
from lab7 import evaluate, create_ui
metrics, model = evaluate('data/shakespeare.txt', output='results', count_threshold=2, k=0.5, seed=42, web_model_path='docs/model.json')
import pandas as pd
display(pd.DataFrame(metrics['perplexities']))
print('Найкраща за перплексією:', metrics['best_model'])
print('Словник:', metrics['vocabulary_size'], 'слів')

## 5. Графіки та приклади прогнозування

In [ ]:
from IPython.display import display, Image, Markdown
display(Image(filename='results/perplexity.png'))
display(Image(filename='results/ngram_counts.png'))
import json
samples = json.loads(Path('results/suggestions.json').read_text(encoding='utf-8'))
for s in samples[:6]:
    print(f"Контекст: {s['context']} | модель {s['requested_n']}, фактична {s['used_n']}")
    print('  ', ', '.join(f"{x['word']} ({x['probability']:.4f})" for x in s['suggestions']))

## 6. Інтерактивний інтерфейс ipywidgets
Введіть англійські слова, за потреби виберіть порядок моделі й початкові літери, натисніть **Запропонувати**, потім кнопку одного з варіантів. Вона додасть вибране слово до введеного тексту.

In [ ]:
widgets_ui = create_ui(model)

## 7. Підсумковий звіт і результати для GitHub
Файли: `REPORT.md`, `metrics.json`, `perplexity.csv`, `suggestions.json`, два PNG-графіки, сайт `docs/`. Наостанок формується ZIP. Публікуйте код і підсумкові результати в GitHub.

In [ ]:
print(Path('results/REPORT.md').read_text(encoding='utf-8')[:4000])

In [ ]:
from pathlib import Path
import zipfile
archive = Path('LR7_results_for_GitHub.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in [Path('results'), Path('docs')]:
        for path in folder.rglob('*'):
            if path.is_file(): z.write(path, path.as_posix())
print('Створено', archive, '|', round(archive.stat().st_size/1024,1), 'КБ')
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Архів доступний у робочій директорії Jupyter Notebook.')

### Висновок
Усі N-грамні моделі від 1 до 5 реалізовані. Зважайте, що на невеликому корпусі довші контексти можуть мати **вищу** перплексію через розрідженість, особливо за простого Add-k-згладжування. Результати зберігаються в `results/REPORT.md`.